# FIFA World Cup 2026 — Linear Regression 2.1
## 01. Data Preparation

**Objective:** Prepare the supplied match-level dataset for predicting **home-team goal difference** (`goal_difference`) from **exactly eight pre-match explanatory variables**.

**Source:** `Linear Regression 2.1(2).xlsx` (supplied workbook; not a CSV). Run the notebook in the folder containing this file, or adjust the path below.

**Scope:** Data preparation only. Exploratory analysis, train/test splitting and modelling will be handled in later notebooks. Existing historical predictors are retained rather than recalculated from information not supplied in this workbook.

## Step 1 — Load and inspect the dataset

In [14]:
# Import libraries needed for preparation
from pathlib import Path
import pandas as pd
import numpy as np

#Show all columns in the output
pd.set_option("display.max_columns", None)
# Load the dataset from the specified CSV file
file_path = "Linear Regression 2.csv"

df = pd.read_csv(file_path, encoding="cp1252")
print("Dataset loaded:", file_path)
print("Shape (rows, columns):", df.shape)

Dataset loaded: Linear Regression 2.csv
Shape (rows, columns): (104, 15)


In [15]:
# Inspect columns and first observations
print(df.columns.tolist())
display(df.head(10))

['match_id', 'date', 'Round_Number', 'Home_Team', 'Away_Team', 'Result', 'Team_prior_matches', 'Opponent_prior_matches', 'Team_prior_goals_scored_avg', 'Opponent_prior_goals_scored_avg', 'Team_rest_days', 'Opponent_rest_days', 'Team_prior_win_rate', 'Opponent_prior_win_rate', 'goal_difference']


,match_id,date,Round_Number,Home_Team,Away_Team,Result,Team_prior_matches,Opponent_prior_matches,Team_prior_goals_scored_avg,Opponent_prior_goals_scored_avg,Team_rest_days,Opponent_rest_days,Team_prior_win_rate,Opponent_prior_win_rate,goal_difference
0,1,2026-06-11 19:00:00.000002931,1,Mexico,South Africa,2 - 0,0,0,0.0,0.0,0,0,0.0,0.0,2
1,2,2026-06-12 01:59:59.999997068,1,Korea Republic,Czechia,2 - 1,0,0,0.0,0.0,0,0,0.0,0.0,1
2,3,2026-06-12 19:00:00.000002931,1,Canada,Bosnia and Herzegovina,1 - 1,0,0,0.0,0.0,0,0,0.0,0.0,0
3,4,2026-06-13 01:00:00.000002931,1,USA,Paraguay,4 - 1,0,0,0.0,0.0,0,0,0.0,0.0,3
4,8,2026-06-13 19:00:00.000002931,1,Qatar,Switzerland,1 - 1,0,0,0.0,0.0,0,0,0.0,0.0,0
5,7,2026-06-13 22:00:00.000002931,1,Brazil,Morocco,1 - 1,0,0,0.0,0.0,0,0,0.0,0.0,0
6,5,2026-06-14 01:00:00.000002931,1,Haiti,Scotland,0 - 1,0,0,0.0,0.0,0,0,0.0,0.0,-1
7,6,2026-06-14 04:00:00.000002931,1,Australia,Türkiye,2 - 0,0,0,0.0,0.0,0,0,0.0,0.0,2
8,10,2026-06-14 16:59:59.999997068,1,Germany,Curaçao,7 - 1,0,0,0.0,0.0,0,0,0.0,0.0,6
9,11,2026-06-14 19:59:59.999997068,1,Netherlands,Japan,2 - 2,0,0,0.0,0.0,0,0,0.0,0.0,0


In [16]:
# Check observation and match counts
print("Rows:", len(df))
print("Unique match IDs:", df["match_id"].nunique())
print("Repeated match IDs:", df["match_id"].duplicated().sum())
print("Missing values by column:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Data types:\n", df.dtypes)

Rows: 104
Unique match IDs: 104
Repeated match IDs: 0
Missing values by column:
 match_id                           0
date                               0
Round_Number                       0
Home_Team                          0
Away_Team                          0
Result                             0
Team_prior_matches                 0
Opponent_prior_matches             0
Team_prior_goals_scored_avg        0
Opponent_prior_goals_scored_avg    0
Team_rest_days                     0
Opponent_rest_days                 0
Team_prior_win_rate                0
Opponent_prior_win_rate            0
goal_difference                    0
dtype: int64
Duplicate rows: 0
Data types:
 match_id                             int64
date                                   str
Round_Number                           str
Home_Team                              str
Away_Team                              str
Result                                 str
Team_prior_matches                   int64
Opponent_prior_matc

In [17]:
# Inspect team names and values that may need cleaning
for column in ["Home_Team", "Away_Team"]:
    values = df[column].dropna().astype(str)
    flagged = [repr(x) for x in values.unique() if x != x.strip() or "\xa0" in x]
    print(column, "unique teams:", values.nunique(), "suspicious names:", flagged)
print("Rounds:", df["Round_Number"].unique())
print("Sample results:", df["Result"].head().tolist())

Home_Team unique teams: 48 suspicious names: []
Away_Team unique teams: 48 suspicious names: []
Rounds: <StringArray>
[             '1',              '2',              '3',    'Round of 32',
    'Round of 16', 'Quarter Finals',    'Semi Finals',         'Finals']
Length: 8, dtype: str
Sample results: ['2 - 0', '2 - 1', '1 - 1', '4 - 1', '1 - 1']


## Step 2 — Data cleaning and validation

Clean textual fields without changing numerical predictor values. Check whether the score recorded in `Result` agrees with the target `goal_difference`. Do not silently discard or overwrite unexpected values.

In [18]:
# Work on a separate copy
clean = df.copy()
for column in ["Home_Team", "Away_Team", "Round_Number", "Result"]:
    clean[column] = (clean[column].astype("string")
                     .str.replace("\xa0", " ", regex=False)
                     .str.strip()
                     .str.replace(r"\s+", " ", regex=True))

# Parse match dates, allowing for fractional seconds in the source file
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
print("Unparseable dates:", clean["date"].isna().sum())
print("Duplicate match IDs:", clean["match_id"].duplicated().sum())
print("Missing values:\n", clean.isna().sum())

Unparseable dates: 0
Duplicate match IDs: 0
Missing values:
 match_id                           0
date                               0
Round_Number                       0
Home_Team                          0
Away_Team                          0
Result                             0
Team_prior_matches                 0
Opponent_prior_matches             0
Team_prior_goals_scored_avg        0
Opponent_prior_goals_scored_avg    0
Team_rest_days                     0
Opponent_rest_days                 0
Team_prior_win_rate                0
Opponent_prior_win_rate            0
goal_difference                    0
dtype: int64


In [19]:
# Extract home and away scores from the existing Result field for validation only
score_parts = clean["Result"].str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$")
score_valid = score_parts.notna().all(axis=1)
parsed_home = pd.to_numeric(score_parts[0], errors="coerce")
parsed_away = pd.to_numeric(score_parts[1], errors="coerce")
score_difference = parsed_home - parsed_away
score_mismatch = score_valid & (score_difference != clean["goal_difference"])
print("Invalid score formats:", (~score_valid).sum())
print("Goal-difference mismatches:", score_mismatch.sum())
if score_mismatch.any():
    display(clean.loc[score_mismatch, ["match_id", "Result", "goal_difference"]])
assert score_valid.all(), "Some scores cannot be parsed. Inspect Result."
assert not score_mismatch.any(), "Some goal differences disagree with Result."
assert clean["match_id"].is_unique, "Match IDs should uniquely identify matches."
assert clean["date"].notna().all(), "Some match dates could not be parsed."

Invalid score formats: 0
Goal-difference mismatches: 0


## Step 3 — Define the target and eight explanatory variables

The target is the **home team’s goal difference** (home goals minus away goals). The `Team_` fields describe the home team, and `Opponent_` fields describe the away team. Match identifiers, scores, dates, round and team names are **not** predictors.

In [20]:
target = "goal_difference"
features = [
    "Team_prior_matches",
    "Opponent_prior_matches",
    "Team_prior_goals_scored_avg",
    "Opponent_prior_goals_scored_avg",
    "Team_rest_days",
    "Opponent_rest_days",
    "Team_prior_win_rate",
    "Opponent_prior_win_rate",
]
assert len(features) == 8 and len(set(features)) == 8
assert all(column in clean.columns for column in features + [target])
print("Target:", target)
print("Eight explanatory variables:")
for i, name in enumerate(features, start=1):
    print(f"X{i}: {name}")

Target: goal_difference
Eight explanatory variables:
X1: Team_prior_matches
X2: Opponent_prior_matches
X3: Team_prior_goals_scored_avg
X4: Opponent_prior_goals_scored_avg
X5: Team_rest_days
X6: Opponent_rest_days
X7: Team_prior_win_rate
X8: Opponent_prior_win_rate


## Step 4 — Verify existing pre-match features

Unlike the reference notebook, this workbook already contains the eight historical variables. We **do not reconstruct** international-match histories or calculate rest days from scratch because the underlying international fixtures are not supplied. Instead, check types, sensible ranges, and the first-match convention. **A zero rest-days value for a team with no prior recorded matches should be treated as a source convention or missing-history placeholder, not necessarily a literal zero-day rest interval.**

**Pre-match integrity still needs external confirmation:** verify the source match history, the rolling-window definitions, and that only fixtures occurring before each target match were used. This notebook does not make that unsupported claim.

In [21]:
# Convert predictors and target to numeric; report unexpected values
for column in features + [target]:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")
print("Missing numeric values:\n", clean[features + [target]].isna().sum())
print("Summary of predictors and target:")
display(clean[features + [target]].describe().T)
assert clean[features + [target]].notna().all().all(), "Review missing or nonnumeric values."

Missing numeric values:
 Team_prior_matches                 0
Opponent_prior_matches             0
Team_prior_goals_scored_avg        0
Opponent_prior_goals_scored_avg    0
Team_rest_days                     0
Opponent_rest_days                 0
Team_prior_win_rate                0
Opponent_prior_win_rate            0
goal_difference                    0
dtype: int64
Summary of predictors and target:


,count,mean,std,min,25%,50%,75%,max
Team_prior_matches,104.0,1.903846,1.663540,0.0,1.0,2.000000,3.00,7.0
Opponent_prior_matches,104.0,1.903846,1.663540,0.0,1.0,2.000000,3.00,7.0
Team_prior_goals_scored_avg,104.0,1.362431,1.285058,0.0,0.0,1.000000,2.25,7.0
Opponent_prior_goals_scored_avg,104.0,1.296291,1.142627,0.0,0.0,1.000000,2.00,5.0
Team_rest_days,104.0,4.067308,2.466334,0.0,3.0,5.000000,6.00,9.0
Opponent_rest_days,104.0,4.048077,2.458904,0.0,3.0,5.000000,6.00,8.0
Team_prior_win_rate,104.0,0.365201,0.409446,0.0,0.0,0.000000,0.75,1.0
Opponent_prior_win_rate,104.0,0.330746,0.369031,0.0,0.0,0.333333,0.60,1.0
goal_difference,104.0,0.480769,2.024034,-4.0,-1.0,0.000000,2.00,6.0


In [22]:
# Validate ranges that follow directly from variable definitions
checks = {
    "negative prior match counts": ((clean["Team_prior_matches"] < 0) | (clean["Opponent_prior_matches"] < 0)).sum(),
    "negative historical goal averages": ((clean["Team_prior_goals_scored_avg"] < 0) | (clean["Opponent_prior_goals_scored_avg"] < 0)).sum(),
    "negative rest days": ((clean["Team_rest_days"] < 0) | (clean["Opponent_rest_days"] < 0)).sum(),
    "win rates outside 0–1": ((~clean["Team_prior_win_rate"].between(0, 1)) | (~clean["Opponent_prior_win_rate"].between(0, 1))).sum(),
    "same team on both sides": (clean["Home_Team"] == clean["Away_Team"]).sum(),
}
print(pd.Series(checks))
assert all(value == 0 for value in checks.values()), "Review out-of-range values."

# First-observation conventions: descriptive check, not proof of historical correctness
for side in ["Team", "Opponent"]:
    first = clean[f"{side}_prior_matches"] == 0
    print(f"{side}: {first.sum()} observations with no prior recorded matches")
    print("  Nonzero win rate with zero prior matches:", (first & (clean[f"{side}_prior_win_rate"] != 0)).sum())
    print("  Nonzero goal average with zero prior matches:", (first & (clean[f"{side}_prior_goals_scored_avg"] != 0)).sum())

negative prior match counts          0
negative historical goal averages    0
negative rest days                   0
win rates outside 0–1                0
same team on both sides              0
dtype: int64
Team: 24 observations with no prior recorded matches
  Nonzero win rate with zero prior matches: 0
  Nonzero goal average with zero prior matches: 0
Opponent: 24 observations with no prior recorded matches
  Nonzero win rate with zero prior matches: 0
  Nonzero goal average with zero prior matches: 0


In [23]:
# Check the target against the actual scores (not used as predictors)
print("Goal-difference range:", clean[target].min(), "to", clean[target].max())
print("Home wins:", (clean[target] > 0).sum())
print("Draws:", (clean[target] == 0).sum())
print("Home losses:", (clean[target] < 0).sum())
print("Chronological order in original workbook:", clean["date"].is_monotonic_increasing)

Goal-difference range: -4 to 6
Home wins: 50
Draws: 24
Home losses: 30
Chronological order in original workbook: True


## Step 5 — Create and export the final preparation dataset

Keep identifiers for tracking and later chronological splitting, the eight predictors, and the target. Do not include `Result` or current-match goals among the predictors. The final file retains all supplied observations.

In [24]:
# Preserve traceability fields without treating them as explanatory variables
id_columns = ["match_id", "date", "Round_Number", "Home_Team", "Away_Team"]
final_columns = id_columns + features + [target]
final_model = clean[final_columns].copy()
final_model = final_model.sort_values(["date", "match_id"], kind="stable").reset_index(drop=True)
print("Final shape:", final_model.shape)
print("Rows:", len(final_model), "| unique matches:", final_model["match_id"].nunique())
print("Predictors:", len(features))
display(final_model.head(10))
assert len(final_model) == len(df)
assert len(features) == 8
assert final_model[features + [target]].notna().all().all()

Final shape: (104, 14)
Rows: 104 | unique matches: 104
Predictors: 8


,match_id,date,Round_Number,Home_Team,Away_Team,Team_prior_matches,Opponent_prior_matches,Team_prior_goals_scored_avg,Opponent_prior_goals_scored_avg,Team_rest_days,Opponent_rest_days,Team_prior_win_rate,Opponent_prior_win_rate,goal_difference
0,1,2026-06-11 19:00:00.000002931,1,Mexico,South Africa,0,0,0.0,0.0,0,0,0.0,0.0,2
1,2,2026-06-12 01:59:59.999997068,1,Korea Republic,Czechia,0,0,0.0,0.0,0,0,0.0,0.0,1
2,3,2026-06-12 19:00:00.000002931,1,Canada,Bosnia and Herzegovina,0,0,0.0,0.0,0,0,0.0,0.0,0
3,4,2026-06-13 01:00:00.000002931,1,USA,Paraguay,0,0,0.0,0.0,0,0,0.0,0.0,3
4,8,2026-06-13 19:00:00.000002931,1,Qatar,Switzerland,0,0,0.0,0.0,0,0,0.0,0.0,0
5,7,2026-06-13 22:00:00.000002931,1,Brazil,Morocco,0,0,0.0,0.0,0,0,0.0,0.0,0
6,5,2026-06-14 01:00:00.000002931,1,Haiti,Scotland,0,0,0.0,0.0,0,0,0.0,0.0,-1
7,6,2026-06-14 04:00:00.000002931,1,Australia,Türkiye,0,0,0.0,0.0,0,0,0.0,0.0,2
8,10,2026-06-14 16:59:59.999997068,1,Germany,Curaçao,0,0,0.0,0.0,0,0,0.0,0.0,6
9,11,2026-06-14 19:59:59.999997068,1,Netherlands,Japan,0,0,0.0,0.0,0,0,0.0,0.0,0


In [ ]:
# Export for the next notebook (EDA)
output_file = Path("Regression_2_1_Prepared_Data.csv")
final_model.to_csv(output_file, index=False)
print("Saved:", output_file.resolve())

Saved: C:\Users\jelag\OneDrive\Documents\MSC SPATIAL SCIENCE\FOUNDATIONS OF DATA SCIENCE\Final 2.1\Regression_2_1_Prepared_Data.csv


: 

## Final Data Preparation Summary

The supplied match-level workbook was inspected, cleaned and validated. The prepared file retains **104 matches**, **eight explanatory variables**, and the goal-difference target, along with identification fields. The scores were checked against goal differences; existing pre-match statistics were retained rather than invented or recomputed without their source records.

**Limitations to document:** The uploaded workbook does not provide the original international-match records used to generate the eight historical predictors. Their timing and rolling-window definitions must be independently verified before claiming that all predictor values were available before kick-off. The next notebook will conduct EDA using `Regression_2_1_Prepared_Data.csv`.